In [1]:
# Install required libraries
!pip -q install pandas numpy scikit-learn transformers datasets accelerate torch seaborn matplotlib tqdm

In [2]:
# Imports and reproducibility setup
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to: {SEED}")

Random seed set to: 42


In [3]:
# Mount Google Drive and define I/O paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/multisocial_outputs')
INPUT_FILES = {
    'en': BASE_DIR / 'multisocial_micro_en.csv',
    'vi': BASE_DIR / 'multisocial_micro_vi_v2.csv',
    'zh': BASE_DIR / 'multisocial_micro_zh.csv',
    'ar': BASE_DIR / 'multisocial_micro_ar.csv',
}

TRAIN_OUT = BASE_DIR / 'multisocial_train.csv'
TEST_OUT = BASE_DIR / 'multisocial_test.csv'

EXPECTED_COLS = [
    'text',
    'label',
    'multi_label',
    'split',
    'language',
    'length',
    'source',
    'potential_noise',
]

print(f'Base directory: {BASE_DIR}')
assert BASE_DIR.exists(), f'Base directory does not exist: {BASE_DIR}'

Mounted at /content/drive
Base directory: /content/drive/MyDrive/multisocial_outputs


In [4]:
# Verify all required input files exist
missing_paths = [str(path) for path in INPUT_FILES.values() if not path.exists()]
if missing_paths:
    missing_msg = '\n'.join(missing_paths)
    raise FileNotFoundError(f'Missing required input CSV files:\n{missing_msg}')

print('All required input files were found.')
assert len(INPUT_FILES) == 4, 'Expected exactly 4 language input files.'

All required input files were found.


In [5]:
# Load and validate input CSVs with progress bar
dfs = []
for lang, fpath in tqdm(INPUT_FILES.items(), desc='Loading CSVs'):
    df = pd.read_csv(fpath)

    missing_cols = set(EXPECTED_COLS) - set(df.columns)
    if missing_cols:
        raise ValueError(f'File {fpath.name} is missing columns: {sorted(missing_cols)}')

    df = df[EXPECTED_COLS].copy()
    df = df.dropna(subset=['text', 'label', 'language'])
    df['text'] = df['text'].astype(str)
    df['label'] = df['label'].astype(int)
    df['language'] = df['language'].astype(str).str.lower().str.strip()

    assert not df.empty, f'Input dataframe is empty after cleanup: {fpath}'
    assert set(df['label'].unique()).issubset({0, 1}), f'Labels must be 0/1 in {fpath.name}'

    # Force consistent language tag from filename to avoid accidental drift.
    df['language'] = lang

    print(f'Loaded {fpath.name}: {len(df)} rows')
    dfs.append(df)

assert len(dfs) == 4, 'Expected exactly 4 loaded dataframes.'
print('All language CSVs loaded and validated.')

Loading CSVs:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded multisocial_micro_en.csv: 4000 rows
Loaded multisocial_micro_vi_v2.csv: 3986 rows
Loaded multisocial_micro_zh.csv: 4000 rows
Loaded multisocial_micro_ar.csv: 4000 rows
All language CSVs loaded and validated.


In [6]:
# Concatenate all language data into a master dataframe
master_df = pd.concat(dfs, ignore_index=True)

assert not master_df.empty, 'Master dataframe is empty after concatenation.'
assert set(master_df['label'].unique()).issubset({0, 1}), 'Labels must be binary (0/1).'

valid_splits = {'train', 'test'}
split_values = set(master_df['split'].astype(str).str.lower().str.strip().unique())
assert split_values.issubset(valid_splits), f'Unexpected split values found: {sorted(split_values)}'

print(f'Master rows: {len(master_df)}')
print('Rows per language in master:')
print(master_df['language'].value_counts())
print('Rows per split in master:')
print(master_df['split'].value_counts())

Master rows: 15986
Rows per language in master:
language
en    4000
zh    4000
ar    4000
vi    3986
Name: count, dtype: int64
Rows per split in master:
split
train    12789
test      3197
Name: count, dtype: int64


In [7]:
# Build train/test from pre-existing split column
train_df = master_df[master_df['split'].astype(str).str.lower().str.strip() == 'train'].copy()
test_df = master_df[master_df['split'].astype(str).str.lower().str.strip() == 'test'].copy()

assert not train_df.empty, 'Train dataframe is empty after filtering by split.'
assert not test_df.empty, 'Test dataframe is empty after filtering by split.'

print(f'Train rows: {len(train_df)}')
print(f'Test rows: {len(test_df)}')
print(f'Overall rows: {len(master_df)}')

Train rows: 12789
Test rows: 3197
Overall rows: 15986


In [8]:
# Save outputs and verify split ratios
train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

assert TRAIN_OUT.exists(), f'Failed to save train file: {TRAIN_OUT}'
assert TEST_OUT.exists(), f'Failed to save test file: {TEST_OUT}'

overall_total = len(master_df)
overall_train = len(train_df)
overall_test = len(test_df)

train_ratio = overall_train / overall_total
test_ratio = overall_test / overall_total
print(f'Train ratio: {train_ratio:.3f}')
print(f'Test ratio: {test_ratio:.3f}')

train_counts = train_df.groupby(['language', 'label']).size().rename('train_count')
test_counts = test_df.groupby(['language', 'label']).size().rename('test_count')
total_counts = master_df.groupby(['language', 'label']).size().rename('total_count')

ratio_table = pd.concat([total_counts, train_counts, test_counts], axis=1).fillna(0).astype(int).reset_index()

print('Counts by language and label (train/test):')
print(ratio_table.to_string(index=False))

assert (ratio_table['train_count'] > 0).all(), 'At least one language-label group has zero train rows.'
assert (ratio_table['test_count'] > 0).all(), 'At least one language-label group has zero test rows.'

print(f'Saved train CSV: {TRAIN_OUT}')
print(f'Saved test CSV: {TEST_OUT}')
print('Notebook 0 verification passed.')

Train ratio: 0.800
Test ratio: 0.200
Counts by language and label (train/test):
language  label  total_count  train_count  test_count
      ar      0         2000         1600         400
      ar      1         2000         1600         400
      en      0         2000         1600         400
      en      1         2000         1600         400
      vi      0         1993         1577         416
      vi      1         1993         1612         381
      zh      0         2000         1600         400
      zh      1         2000         1600         400
Saved train CSV: /content/drive/MyDrive/multisocial_outputs/multisocial_train.csv
Saved test CSV: /content/drive/MyDrive/multisocial_outputs/multisocial_test.csv
Notebook 0 verification passed.
